# 量化交易入门 Vol.2：Alpha 因子工程

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/greathousesh/qlora-sft-tutorial/blob/main/quant/02_alpha_factors.ipynb)

> **Kaggle 一键运行**：点击上方按钮 → 选择 **"CPU"** → Run All

## 本节你将学到

| 知识点 | 说明 |
|--------|------|
| **Alpha 因子的定义** | 因子 = 对未来收益的预测信号 |
| **经典因子族** | 动量、反转、波动率、成交量、质量因子 |
| **IC 分析** | 用信息系数定量评估因子有效性 |
| **因子衰减** | 因子信号的时效性分析 |
| **分位数分析** | 看因子能否区分强弱股票 |
| **qlib 表达式引擎** | 用 qlib 的 DSL 高效定义因子 |

## 接续 Vol.1

本 notebook 直接从 yfinance 重新下载数据，无需先运行 Vol.1。

In [ ]:
import subprocess, sys
pkgs = ["yfinance>=0.2.30", "pandas>=1.5.0", "numpy>=1.23.0",
        "matplotlib>=3.6.0", "seaborn>=0.12.0", "scipy>=1.9.0"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
print("✅ 依赖安装完毕")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy.stats import spearmanr, pearsonr

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11,
                     'axes.titlesize': 12, 'axes.grid': True, 'grid.alpha': 0.3})

TICKERS   = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'TSLA', 'JPM', 'GS', 'JNJ', 'WMT']
BENCHMARK = 'SPY'
START, END = '2018-01-01', '2024-01-01'

raw      = yf.download(TICKERS + [BENCHMARK], start=START, end=END,
                        auto_adjust=True, progress=False)
prices   = raw['Close'].dropna(how='all')
volumes  = raw['Volume'].dropna(how='all')
highs    = raw['High'].dropna(how='all')
lows     = raw['Low'].dropna(how='all')
log_rets = np.log(prices / prices.shift(1)).dropna()

print(f"数据加载完毕: {prices.shape[0]} 交易日 × {len(TICKERS)} 只股票 + SPY")

## Step 1：什么是 Alpha 因子？

### 从数据到信号

**Alpha 因子（Alpha Factor）** 是一个数值，它在某种程度上能预测股票未来的相对收益。

```
因子值（今天计算）→ 预测 → 股票未来收益率（相对于其他股票）

例子：
  动量因子 = 过去 12 个月收益率
  → 过去涨得多的股票，未来可能继续跑赢（动量效应）

  反转因子 = 过去 1 个月收益率（取反）
  → 上个月跌得多的股票，下个月可能反弹（均值回归）
```

### 因子有效性的前提：市场的"有限"有效

有效市场假说（EMH）说所有信息已反映在价格中，因子不应有效。
但实证研究发现很多因子长期有效，原因可能是：
1. 投资者行为偏差（过度反应/反应不足）
2. 风险溢价（持有某类股票需要额外补偿）
3. 结构性约束（机构有持仓限制、流动性约束）

### 因子的基本分类

| 类别 | 代表因子 | 直觉 |
|------|----------|------|
| 动量（Momentum）| 过去 12 个月收益 | 趋势延续 |
| 反转（Reversal）| 过去 1 个月收益（取反）| 短期过度反应 |
| 低波动（Low Vol）| 过去波动率（取反）| 低风险溢价 |
| 成交量（Volume）| 成交量变化 | 资金流入/流出 |
| 质量（Quality）| ROE、盈利稳定性 | 好公司溢价 |

## Step 2：手工实现经典因子

**两个关键注意事项**：
1. **防止未来数据泄漏**：计算因子时，只能用到「昨天及之前」的数据
2. **截面中性化**：每天对所有股票做 rank 或 zscore，消除市场共同波动

我们将实现 5 个经典因子族：
- **F1**: 动量因子（12-1 月）
- **F2**: 短期反转（1 月）  
- **F3**: 低波动因子
- **F4**: 成交量动量
- **F5**: 价格强度（RSI 类）

In [ ]:
def zscore_cs(df):
    """截面 Z-score 标准化（每行减均值除标准差）"""
    return df.sub(df.mean(axis=1), axis=0).div(df.std(axis=1), axis=0)

def rank_cs(df):
    """截面排名（每行，rank 0~1）"""
    return df.rank(axis=1, pct=True)

# ─────────────────────────────────────────────────────────
# F1：12-1 月动量因子（经典 Jegadeesh-Titman 动量）
# 用过去 252 个交易日的收益，跳过最近 21 天（避免短期反转污染）
# ─────────────────────────────────────────────────────────
prices_stocks = prices[TICKERS]

ret_252 = prices_stocks.pct_change(252)   # 过去 252 天总收益
ret_21  = prices_stocks.pct_change(21)    # 过去 21 天收益（要剔除）
F1_raw  = ret_252 - ret_21               # 12-1 月动量
F1      = rank_cs(F1_raw.shift(1))       # shift(1) 确保用昨天数据

# ─────────────────────────────────────────────────────────
# F2：1 个月短期反转因子（取负号：过去涨得多的反转下跌）
# ─────────────────────────────────────────────────────────
F2_raw = -prices_stocks.pct_change(21)   # 取负：反转逻辑
F2     = rank_cs(F2_raw.shift(1))

# ─────────────────────────────────────────────────────────
# F3：低波动因子（过去 60 天波动率取反）
# 低波动股票通常长期表现好（低波动异象）
# ─────────────────────────────────────────────────────────
vol_60  = log_rets[TICKERS].rolling(60).std() * np.sqrt(252)  # 年化波动率
F3_raw  = -vol_60   # 取负：波动率低的股票排名高
F3      = rank_cs(F3_raw.shift(1))

# ─────────────────────────────────────────────────────────
# F4：成交量变化因子（近期成交量 vs 长期成交量）
# 成交量放大 + 价格上涨 = 强势信号
# ─────────────────────────────────────────────────────────
vol_stocks = volumes[TICKERS]
vol_ratio  = vol_stocks.rolling(5).mean() / vol_stocks.rolling(60).mean()  # 短期/长期成交量比
ret_5d     = prices_stocks.pct_change(5)                                    # 5 日收益率方向
F4_raw     = vol_ratio * np.sign(ret_5d)  # 放量上涨=正，放量下跌=负
F4         = rank_cs(F4_raw.shift(1))

# ─────────────────────────────────────────────────────────
# F5：RSI（相对强弱指数）因子
# RSI < 30 超卖（可能反弹），RSI > 70 超买（可能回调）
# 作为因子：低 RSI → 预期未来收益高（反转逻辑）
# ─────────────────────────────────────────────────────────
def compute_rsi(prices_df, window=14):
    delta = prices_df.diff()
    gain  = delta.clip(lower=0)
    loss  = (-delta).clip(lower=0)
    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()
    rs  = avg_gain / avg_loss.replace(0, 1e-10)
    rsi = 100 - 100 / (1 + rs)
    return rsi

rsi_14 = compute_rsi(prices_stocks, window=14)
F5_raw = 100 - rsi_14   # 100-RSI：RSI 越低（超卖）→ 因子值越高
F5     = rank_cs(F5_raw.shift(1))

factors = {'F1_动量12-1月': F1, 'F2_短期反转': F2, 'F3_低波动': F3,
           'F4_成交量动量': F4, 'F5_反向RSI': F5}

print("所有因子计算完毕：")
for name, f in factors.items():
    valid = f.notna().sum().sum()
    print(f"  {name}: shape={f.shape}, 有效值={valid}")

## Step 3：因子评估 —— IC 分析

**IC（Information Coefficient，信息系数）** 是评估因子有效性最核心的指标：

$$IC_t = \text{SpearmanCorr}(\text{Factor}_{t}, \text{Return}_{t+N})$$

也就是：今天的因子截面排名，与未来 N 天收益截面排名的相关系数。

| 指标 | 计算 | 含义 |
|------|------|------|
| **IC 均值** | mean(IC_t) | 因子平均预测能力，>0.02 认为有效 |
| **IC 标准差** | std(IC_t) | IC 的稳定性，越小越好 |
| **ICIR** | mean/std | 信噪比，>0.5 认为稳定有效 |
| **IC>0 占比** | P(IC>0) | 因子正向预测的比例，>50% 为正向因子 |

In [ ]:
HOLDING = 10  # 预测 10 个交易日后的收益

def compute_ic_series(factor_df, returns_df, forward_days=HOLDING):
    """计算每天的截面 Rank IC 序列"""
    future_ret = returns_df.rolling(forward_days).sum().shift(-forward_days)
    ic_list = []
    for date in factor_df.index:
        f = factor_df.loc[date].dropna()
        r = future_ret.loc[date] if date in future_ret.index else pd.Series(dtype=float)
        common = f.index.intersection(r.dropna().index)
        if len(common) >= 5:
            corr, _ = spearmanr(f[common], r[common])
            ic_list.append({'date': date, 'IC': corr})
    return pd.DataFrame(ic_list).set_index('date')['IC']

# 只在 2019 年后计算（因子需要足够历史数据）
eval_start = '2019-01-01'
rets_eval  = log_rets[TICKERS][eval_start:]

print(f"计算各因子 IC（预测 {HOLDING} 日后收益）...")
ic_results = {}
for name, factor in factors.items():
    factor_eval = factor[eval_start:]
    ic_series   = compute_ic_series(factor_eval, rets_eval, HOLDING)
    ic_results[name] = ic_series
    print(f"  {name}: IC均值={ic_series.mean():.4f}, ICIR={ic_series.mean()/ic_series.std():.3f}, IC>0={( ic_series>0).mean()*100:.1f}%")

print("\n完成！")

In [ ]:
# IC 分析可视化
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
axes = axes.flatten()

for ax, (name, ic_s) in zip(axes[:5], ic_results.items()):
    ic_roll = ic_s.rolling(20).mean()
    
    ax.bar(ic_s.index, ic_s, alpha=0.3, color='steelblue', width=1)
    ax.plot(ic_roll.index, ic_roll, color='steelblue', lw=2, label='20日均值')
    ax.axhline(0, color='black', lw=1)
    ax.axhline(ic_s.mean(), color='red', ls='--', lw=1.5,
               label=f'均值={ic_s.mean():.4f}')
    
    icir = ic_s.mean() / ic_s.std()
    pct_pos = (ic_s > 0).mean() * 100
    ax.set_title(f'{name}\nICIR={icir:.3f}  |  IC>0占比: {pct_pos:.1f}%')
    ax.set_ylabel('Rank IC')
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# 第 6 格：各因子 IC 均值 + ICIR 对比条形图
ax = axes[5]
names = list(ic_results.keys())
ic_means = [ic_results[n].mean() for n in names]
icirs    = [ic_results[n].mean() / ic_results[n].std() for n in names]

x = np.arange(len(names))
width = 0.35
bars1 = ax.bar(x - width/2, ic_means, width, label='IC 均值', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, icirs,    width, label='ICIR',    color='coral',     alpha=0.8)

ax.axhline(0.02, color='blue', ls='--', lw=1, alpha=0.6, label='IC>0.02 (有效线)')
ax.axhline(0.5,  color='red',  ls='--', lw=1, alpha=0.6, label='ICIR>0.5 (稳定线)')
ax.set_xticks(x)
ax.set_xticklabels([n.split('_')[0] for n in names])
ax.set_title('各因子综合评分')
ax.legend(fontsize=9)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle(f'因子 IC 分析（预测 {HOLDING} 日收益，2019-2023）', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('ic_analysis.png', bbox_inches='tight')
plt.show()

## Step 4：因子衰减分析 —— 信号能持续多久？

**因子衰减（Factor Decay）** 分析回答：因子信号对 1 天后有效？5 天后？20 天后？

如果一个因子的 IC 随预测期延长而快速归零，说明：
- 这是短期信号 → 需要高频换手，交易成本高
- 如果 IC 在 60+ 天仍然有效 → 低频策略可用，成本低

**不同因子族的典型衰减特征**：
- 动量：慢衰减（信号持续 3-12 个月）
- 短期反转：极快衰减（信号只持续几天）
- 低波动：极慢衰减（信号持续数月至数年）

In [ ]:
horizons = [1, 5, 10, 20, 40, 60]  # 预测 1/5/10/20/40/60 天后的收益

decay_results = {name: [] for name in factors}

print("计算因子衰减（每个 horizon 需要约 10 秒）...")
for h in horizons:
    print(f"  horizon = {h} 天...", end=' ')
    for name, factor in factors.items():
        factor_eval = factor[eval_start:]
        ic_s = compute_ic_series(factor_eval, rets_eval, h)
        decay_results[name].append(ic_s.mean())
    print("done")

fig, ax = plt.subplots(figsize=(12, 6))

colors_map = ['steelblue', 'coral', 'green', 'purple', 'orange']
for (name, vals), color in zip(decay_results.items(), colors_map):
    ax.plot(horizons, vals, 'o-', lw=2, ms=8, color=color, label=name)
    ax.fill_between(horizons, 0, vals, alpha=0.06, color=color)

ax.axhline(0, color='black', lw=1.2)
ax.axhline(0.02, color='gray', lw=1, ls='--', alpha=0.7, label='IC=0.02 有效线')
ax.set_xlabel('预测 Horizon（交易日）')
ax.set_ylabel('IC 均值')
ax.set_title('因子 IC 衰减曲线\n斜率平缓=长效信号（适合低频），陡降=短效信号（需高频换手）')
ax.set_xticks(horizons)
ax.legend()

plt.tight_layout()
plt.savefig('ic_decay.png', bbox_inches='tight')
plt.show()

print("\n衰减对比（各 horizon 的 IC 均值）：")
decay_df = pd.DataFrame(decay_results, index=horizons)
decay_df.index.name = 'Horizon'
print(decay_df.round(4).to_string())

## Step 5：分位数收益分析 —— 因子能区分好坏股票吗？

**分位数收益（Quantile Returns）** 是因子有效性的最直观验证：

1. 每天根据因子值将股票分成 5 组（五分位）：Q1（最低）到 Q5（最高）
2. 计算每组的平均收益
3. 如果因子有效：Q5 应该显著好于 Q1（单调递增关系）

**关键指标**：
- **多空收益差** = Q5 - Q1 收益（越大越好）
- **IC 多空比** = 因子在 Q5/Q1 的分布是否清晰分离

In [ ]:
N_QUANTILES = 5
HOLDING_Q   = 10  # 每次持有 10 天

def compute_quantile_returns(factor_df, returns_df, n_q=N_QUANTILES, holding=HOLDING_Q):
    """每天按因子值分组，计算各组未来 holding 天的平均收益"""
    future_ret = returns_df.rolling(holding).sum().shift(-holding)
    
    # 只保留换仓日（每 holding 天换一次）
    dates = factor_df.index[::holding]
    
    q_rets = {q: [] for q in range(1, n_q + 1)}
    q_dates = []
    
    for date in dates:
        if date not in factor_df.index:
            continue
        f = factor_df.loc[date].dropna()
        r = future_ret.loc[date] if date in future_ret.index else pd.Series(dtype=float)
        common = f.index.intersection(r.dropna().index)
        if len(common) < n_q:
            continue
        
        f_sorted = f[common].sort_values()
        group_size = len(f_sorted) // n_q
        
        for q in range(1, n_q + 1):
            start_idx = (q - 1) * group_size
            end_idx   = q * group_size if q < n_q else len(f_sorted)
            stocks_in_q = f_sorted.iloc[start_idx:end_idx].index
            q_rets[q].append(r[stocks_in_q].mean())
        q_dates.append(date)
    
    result = pd.DataFrame(q_rets, index=q_dates)
    result.columns = [f'Q{q}' for q in range(1, n_q + 1)]
    return result

# 只对最有效的 3 个因子做分位数分析（节省时间）
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for col, (name, factor) in enumerate(list(factors.items())[:3]):
    factor_eval = factor[eval_start:]
    q_rets = compute_quantile_returns(factor_eval, rets_eval[TICKERS])
    cum_q  = (1 + q_rets).cumprod()
    
    # 上排：累计收益
    ax = axes[0][col]
    colors_q = ['#d73027', '#fc8d59', '#fee090', '#91bfdb', '#4575b4']
    for q_col, color in zip(cum_q.columns, colors_q):
        ax.plot(cum_q.index, cum_q[q_col], color=color, lw=2, label=q_col)
    ax.set_title(f'{name}\n五分位累计收益')
    ax.set_ylabel('Cumulative Return')
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    
    # 下排：平均收益柱状图
    ax2 = axes[1][col]
    avg_rets = q_rets.mean() * 252 / HOLDING_Q  # 年化
    bar_colors = ['#d73027', '#fc8d59', '#fee090', '#91bfdb', '#4575b4']
    bars = ax2.bar(q_rets.columns, avg_rets * 100, color=bar_colors, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, avg_rets * 100):
        ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
    ax2.axhline(0, color='black', lw=1)
    ax2.set_ylabel('年化平均收益 %')
    ax2.set_title(f'Q5-Q1 多空收益差: {(avg_rets.iloc[-1]-avg_rets.iloc[0])*100:.1f}% / 年')

plt.suptitle('因子分位数分析（五分位，持有10天）\nQ1=因子最低组, Q5=因子最高组  —  好因子应呈现单调递增', 
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('quantile_returns.png', bbox_inches='tight')
plt.show()

## Step 6：因子合成 —— 多因子模型基础

单个因子的 IC 通常很低（0.02-0.05），信号噪音很大。  
**多因子合成**的思想：把多个弱信号组合成一个更强的复合信号。

```
复合因子 = α₁×F1 + α₂×F2 + α₃×F3 + ...
```

最简单的方式是**等权平均**，更高级的方式是用机器学习（Vol.3）。  
组合后的好处：
- IC 更高（信号相互补充）
- IC 更稳定（不同因子在不同市场环境下各有优势）
- 在 Vol.3 中，这些因子将作为 ML 模型的特征

In [ ]:
# 等权合成复合因子（每个因子已经做了截面 rank，范围 0-1）
combined_factor = pd.DataFrame(0.0, index=F1.index, columns=TICKERS)
for name, f in factors.items():
    combined_factor = combined_factor.add(f, fill_value=0)
combined_factor = rank_cs(combined_factor)  # 再做一次截面 rank

# 计算合成因子的 IC
factor_eval = combined_factor[eval_start:]
ic_combined = compute_ic_series(factor_eval, rets_eval, HOLDING)

print("各因子 IC 对比：")
print(f"{'因子':<15}  {'IC均值':>8}  {'ICIR':>8}  {'IC>0%':>8}")
print("-" * 45)
for name, ic_s in ic_results.items():
    print(f"{name:<15}  {ic_s.mean():>8.4f}  {ic_s.mean()/ic_s.std():>8.3f}  {(ic_s>0).mean()*100:>7.1f}%")
icir_combined = ic_combined.mean() / ic_combined.std()
print("-" * 45)
print(f"{'复合因子(等权)':<15}  {ic_combined.mean():>8.4f}  {icir_combined:>8.3f}  {(ic_combined>0).mean()*100:>7.1f}%")

# 可视化：单因子 vs 复合因子 IC 分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
all_ics = list(ic_results.values()) + [ic_combined]
all_names = list(ic_results.keys()) + ['复合因子']
positions = range(1, len(all_ics) + 1)
bp = ax.boxplot(all_ics, labels=[n.split('_')[1] if '_' in n else n for n in all_names],
                patch_artist=True, notch=False,
                medianprops=dict(color='red', linewidth=2))
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
bp['boxes'][-1].set_facecolor('lightcoral')
ax.axhline(0, color='black', lw=1, ls='--')
ax.set_ylabel('Rank IC')
ax.set_title('各因子 IC 分布（箱线图）\n红色=复合因子，理想情况中位数>0')
ax.tick_params(axis='x', rotation=20)

ax2 = axes[1]
for (name, ic_s), color in zip(ic_results.items(), colors_map):
    ic_cum = ic_s.cumsum()
    ax2.plot(ic_cum.index, ic_cum, lw=1.2, alpha=0.7, label=name)
ax2.plot(ic_combined.cumsum().index, ic_combined.cumsum(),
         lw=2.5, color='black', label='复合因子')
ax2.axhline(0, color='gray', lw=0.8)
ax2.set_ylabel('IC 累积和（越高越好）')
ax2.set_title('IC 累积曲线\n斜率=因子持续有效性，复合因子更平稳')
ax2.legend(fontsize=9)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('combined_factor.png', bbox_inches='tight')
plt.show()

## Step 7：qlib 表达式引擎预览

在上面的代码里，我们手动写了很多 pandas 操作来计算因子。  
**qlib 的表达式引擎**让你用类似公式的 DSL（领域特定语言）来定义因子，更简洁、更高效：

```python
# 手动写法（pandas）
momentum = prices.pct_change(252) - prices.pct_change(21)

# qlib 表达式引擎
momentum_expr = "Ref($close, 21) / Ref($close, 252) - 1"
```

qlib 支持的表达式算子包括：

| 算子类型 | 示例 | 说明 |
|----------|------|------|
| 引用 | `$close`, `$volume` | 原始字段 |
| 历史 | `Ref($close, 5)` | N 日前的值 |
| 聚合 | `Mean($close, 20)` | N 日均值 |
| 滚动 | `Std($close, 20)` | N 日标准差 |
| 排名 | `Rank($close, 30)` | N 日内历史排名 |
| 数学 | `Log($close)`, `Abs($ret)` | 数学函数 |

下面是 qlib 中完整的因子定义示例（在 Vol.5 会实际运行）：

In [ ]:
# qlib 因子表达式示例（展示语法，Vol.5 会实际运行）
qlib_factor_expressions = {
    "F1_动量12-1月": "Ref($close,21)/$close - Ref($close,252)/$close",

    "F2_短期反转":  "$close / Ref($close, 21) - 1",  # 取负：后面用 -1 乘

    "F3_低波动":    "Std(Log($close/Ref($close,1)), 60)",  # 60日历史波动率

    "F4_成交量比":  "Mean($volume,5) / Mean($volume,60) * Sign($close/Ref($close,5)-1)",

    "F5_反向RSI":  "100 - (Mean(If($close>Ref($close,1),$close-Ref($close,1),0),14) "
                  "/ (Mean(If($close>Ref($close,1),$close-Ref($close,1),0),14) + "
                  "Mean(If($close<Ref($close,1),Ref($close,1)-$close,0),14)) * 100)",

    # qlib 中也内置了很多算子：
    "高低价振幅":   "($high - $low) / $close",
    "量价背离":     "Corr($close, $volume, 30)",  # 30日价量相关（负相关=量价背离）
    "价格加速度":   "$close/Ref($close,5) - Ref($close,5)/Ref($close,10)",  # 动量加速
}

print("qlib 因子表达式示例：")
print("=" * 65)
for name, expr in qlib_factor_expressions.items():
    print(f"\n{name}:")
    print(f"  {expr}")

print("\n" + "=" * 65)
print("""
实际使用方式（Vol.5 中运行）:

  import qlib
  from qlib.data import D

  qlib.init(provider_uri='~/.qlib/qlib_data/us_data', region='us')

  df = D.features(
      instruments=['AAPL', 'MSFT', 'NVDA'],
      fields=[
          '$close',
          'Ref($close,21)/$close - Ref($close,252)/$close',  # 动量
          'Std(Log($close/Ref($close,1)), 60)',               # 波动率
      ],
      start_time='2020-01-01',
      end_time='2023-01-01'
  )

qlib 会自动处理：对齐、缺失值、并行计算、数据缓存
""")

## 本节总结

```
Alpha 因子工程完整流程

原始数据（OHLCV）
    │
    ├── 定义因子（公式 / qlib表达式）
    │       ├── 动量：趋势延续
    │       ├── 反转：均值回归  
    │       ├── 低波动：低风险溢价
    │       └── 成交量、RSI...
    │
    ├── 因子评估
    │       ├── IC 均值（预测能力）> 0.02 有效
    │       ├── ICIR（稳定性）> 0.5 稳定
    │       └── 分位数分析（单调性）
    │
    ├── 因子衰减分析（信号时效）
    │
    └── 多因子合成（等权 → 下一节用 ML 学习权重）
```

### 因子有效性判断标准

| 指标 | 阈值 | 说明 |
|------|------|------|
| IC 均值 | > 0.02 | 有一定预测能力 |
| ICIR | > 0.5 | 信号稳定 |
| IC>0 占比 | > 55% | 因子方向一致 |
| Q5-Q1 多空差 | > 5% 年化 | 因子可以区分好坏股票 |

## 下一步：Vol.3 机器学习选股

我们已经有了多个因子特征。下一节，用 **LightGBM** 自动学习因子的最优组合权重，构建更强的预测模型。

---
*Vol.2 完 | 课程：量化交易从入门到 qlib*